<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_40_tensorflow_keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🟧 **Module 40 — TensorFlow & Keras (the other half of the deep-learning ecosystem)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)


# Module 40 — TensorFlow & Keras

> Most of the course is PyTorch (M16-M24, M39). But a huge slice of production ML — Google products, mobile apps via **TF Lite**, embedded devices, the entire Keras-3 ecosystem — runs on TensorFlow. **Keras 3** is now backend-agnostic (TF / JAX / PyTorch), but the workflow you'll see at TF shops is still the classic Keras one. This module makes you bilingual.

### What you'll cover
1. TF vs PyTorch — when each wins
2. Tensors, eager mode, `tf.function` (graph mode)
3. **`tf.data`** — input pipelines that don't bottleneck the GPU
4. **Keras Sequential** — quickest path to a working model
5. **Keras Functional** — multi-input, multi-output, branching
6. **Keras Subclassing** — full control (PyTorch-style)
7. Callbacks — early stopping, LR schedules, checkpoints, TensorBoard
8. Save / load — `.keras` files + SavedModel format
9. **TF Lite** — convert + quantise for mobile / edge
10. **TF Serving** — what production deploys actually do


## 1 · TF vs PyTorch — when each wins

| TensorFlow / Keras wins | PyTorch wins |
|---|---|
| Mobile (TF Lite), browser (TF.js), microcontrollers (TF Micro) | Research, frontier model code (LLMs, diffusion) |
| TPU support is first-class | NVIDIA GPU support is first-class |
| Production deploy story (TF Serving, Vertex AI) is mature | HF / vLLM / Unsloth / Triton ecosystem |
| Great for tabular + classic CV / time-series at scale | Most academic papers ship PyTorch reference code |

In 2026 the **modeling code** in both frameworks looks almost identical — Keras 3 even runs *on* PyTorch. Pick TF when your **deployment target** demands it (mobile / edge / TPU / GCP-native).

## 2 · Tensors, eager mode, `tf.function`

In [ ]:
!pip -q install tensorflow

In [ ]:
import tensorflow as tf
print(tf.__version__, "  GPU:", tf.config.list_physical_devices('GPU'))

# Tensors look like numpy arrays
a = tf.constant([[1.,2.],[3.,4.]])
b = tf.constant([[5.,6.],[7.,8.]])
print(a @ b)             # eager: runs immediately like numpy/torch

In [ ]:
# Decorate a Python function with @tf.function to compile it into a static graph.
# First call traces; subsequent calls are fast (no Python overhead).
@tf.function
def fast_op(x, y):
    return tf.reduce_sum(x @ y)

print(fast_op(a, b))     # first call: traces + executes
print(fast_op(a, b))     # second call: pure graph, fast

## 3 · `tf.data` — input pipelines

`tf.data.Dataset` is TF's answer to PyTorch's `DataLoader`. It builds a pipeline that pre-fetches and overlaps CPU + GPU work — the easiest 1.5-3× speedup you can get on a real training loop.

In [ ]:
import numpy as np

xs = np.random.randn(10_000, 32).astype("float32")
ys = (xs.sum(axis=1) > 0).astype("int32")

ds = (tf.data.Dataset.from_tensor_slices((xs, ys))
        .shuffle(1024)
        .batch(64)
        .prefetch(tf.data.AUTOTUNE))      # overlap GPU compute with next batch's prep
print(ds)

## 4 · Keras Sequential — fastest path to MNIST

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0
print(x_train.shape, y_train.shape)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28,28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10),                  # logits
])
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
model.summary()

In [ ]:
model.fit(x_train, y_train, epochs=2, batch_size=128, validation_split=0.1)
model.evaluate(x_test, y_test, verbose=2)

## 5 · Keras Functional — branching, multi-input

In [ ]:
# Tiny example: image branch + tabular branch → concatenate → classify
img_in = tf.keras.Input(shape=(28,28,1), name="image")
tab_in = tf.keras.Input(shape=(5,),       name="tabular")

x = tf.keras.layers.Conv2D(8, 3, activation="relu")(img_in)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
y = tf.keras.layers.Dense(8, activation="relu")(tab_in)
z = tf.keras.layers.Concatenate()([x, y])
out = tf.keras.layers.Dense(10)(z)

multi = tf.keras.Model(inputs={"image": img_in, "tabular": tab_in}, outputs=out)
multi.summary()

## 6 · Keras Subclassing — full control (PyTorch-style)

In [ ]:
class Tiny(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.flat = tf.keras.layers.Flatten()
        self.d1 = tf.keras.layers.Dense(64, activation="relu")
        self.d2 = tf.keras.layers.Dense(10)
    def call(self, x, training=False):
        return self.d2(self.d1(self.flat(x)))

m = Tiny()
m.compile(optimizer="adam",
          loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
          metrics=["accuracy"])
m.fit(x_train, y_train, epochs=1, batch_size=128, verbose=2)

## 7 · Callbacks — the production utilities

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("best.keras", save_best_only=True, monitor="val_loss"),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=1),
    tf.keras.callbacks.TensorBoard(log_dir="logs", histogram_freq=1),
]
# (Pass `callbacks=callbacks` into model.fit(...) — skipped here to keep the demo short.)

## 8 · Save / load

In [ ]:
model.save("mnist.keras")            # native Keras format (single zip)
loaded = tf.keras.models.load_model("mnist.keras")
print(loaded.evaluate(x_test, y_test, verbose=0))

In [ ]:
# SavedModel — language-agnostic, what TF Serving / Vertex AI expect
model.export("mnist_saved")
!ls mnist_saved

## 9 · TF Lite — mobile / edge

`TFLiteConverter` turns a model into a single `.tflite` file you can drop into an Android app or a Raspberry Pi. Optionally **quantise** to int8 for ~4× smaller + faster on CPUs without FPUs.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]   # post-training quantisation
tflite_bytes = converter.convert()
open("mnist.tflite","wb").write(tflite_bytes)
print("size:", len(tflite_bytes), "bytes")

In [ ]:
# Run inference with the .tflite interpreter (the same runtime ships in Android / iOS)
interp = tf.lite.Interpreter(model_content=tflite_bytes); interp.allocate_tensors()
inp = interp.get_input_details()[0]; out = interp.get_output_details()[0]
sample = x_test[:1].astype(inp["dtype"])
interp.set_tensor(inp["index"], sample); interp.invoke()
print("predicted:", interp.get_tensor(out["index"]).argmax(), "  true:", y_test[0])

## 10 · TF Serving — what production looks like

```bash
# host the SavedModel behind a gRPC + REST API, one command:
docker run -p 8501:8501 \
    -v $PWD/mnist_saved:/models/mnist/1 \
    -e MODEL_NAME=mnist \
    tensorflow/serving

# call it:
curl -d '{"instances":[[[...28x28 floats...]]]}' \
    http://localhost:8501/v1/models/mnist:predict
```

| Topic | What to know |
|---|---|
| **Versioning** | drop a new model into `/models/mnist/2/` and TF Serving hot-swaps |
| **Batching** | enable `--enable_batching` to coalesce concurrent requests |
| **gRPC vs REST** | gRPC is ~5-10× faster for big tensors |
| **Vertex AI / SageMaker** | both deploy SavedModels natively |


## ✅ Recap
- TF / Keras shines for **mobile (TF Lite), TPU, GCP-native deploys**.
- Keras has **three APIs**: Sequential (lines), Functional (graphs), Subclassing (full control).
- **`tf.data`** + `prefetch(AUTOTUNE)` keeps the GPU fed.
- **`@tf.function`** compiles Python → static graph for speed.
- **TFLite** for edge, **SavedModel + TF Serving** for the cloud.

Next: **M41 — CUDA basics** (writing your own GPU kernels).
